In [20]:
import pandas as pd
from catboost import CatBoostRegressor, Pool, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from yellowbrick.model_selection import LearningCurve
from yellowbrick.contrib.wrapper import wrap, regressor
import numpy as np

In [2]:
cols = ['bedrooms', 'beds', 'baths', 'luxury_items', 'luxury_score', 'location_desirability']

In [3]:
df = pd.read_parquet('./dataset/forCatboost.parquet')
df = df.loc[:, ['Price'] + cols +['quality', 'city']].astype(np.float64, errors='ignore')
vc = df['city'].value_counts()
vc =  vc[vc >= 14]
cities = vc.index
df = df[df['city'].isin(cities)].reset_index(drop=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Price                  1474 non-null   float64
 1   bedrooms               1474 non-null   float64
 2   beds                   1474 non-null   float64
 3   baths                  1474 non-null   float64
 4   luxury_items           1474 non-null   float64
 5   luxury_score           1474 non-null   float64
 6   location_desirability  1474 non-null   float64
 7   quality                1474 non-null   str    
 8   city                   1474 non-null   str    
dtypes: float64(7), str(2)
memory usage: 123.6 KB


In [6]:
X = df.iloc[:, 1:]
y = df['Price']

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [40]:
p = Pool(X, y, cat_features=['city', 'quality'])
params = {
  'cat_features': ['city', 'quality'],
  'depth': 5,
  'loss_function': 'RMSE',
  'iterations': 400
  
}

cv_res = cv(p, params, fold_count=3)
cv_res

Training on fold [0/3]
0:	learn: 128453.2537547	test: 119301.2812683	best: 119301.2812683 (0)	total: 33.9ms	remaining: 13.5s
1:	learn: 125819.1959283	test: 116816.9177801	best: 116816.9177801 (1)	total: 81.4ms	remaining: 16.2s
2:	learn: 123316.0707483	test: 114355.3270707	best: 114355.3270707 (2)	total: 111ms	remaining: 14.7s
3:	learn: 120901.1271124	test: 112013.5421964	best: 112013.5421964 (3)	total: 156ms	remaining: 15.4s
4:	learn: 118443.8580010	test: 109654.1550124	best: 109654.1550124 (4)	total: 207ms	remaining: 16.4s
5:	learn: 116373.6087114	test: 107537.2331644	best: 107537.2331644 (5)	total: 228ms	remaining: 14.9s
6:	learn: 114125.5352114	test: 105302.2802823	best: 105302.2802823 (6)	total: 275ms	remaining: 15.4s
7:	learn: 111984.0718976	test: 103097.8855772	best: 103097.8855772 (7)	total: 310ms	remaining: 15.2s
8:	learn: 109861.5898058	test: 101171.1630915	best: 101171.1630915 (8)	total: 351ms	remaining: 15.2s
9:	learn: 107903.8285458	test: 99392.5307384	best: 99392.5307384 (

,iterations,test-RMSE-mean,test-RMSE-std,train-RMSE-mean,train-RMSE-std
0,0,125368.565176,9216.216104,125490.139499,4625.354206
1,1,122883.404236,9261.188175,122950.619804,4552.677699
2,2,120372.343635,9278.835013,120436.617054,4505.846097
3,3,118012.185768,9285.513541,118055.872854,4501.891142
4,4,115733.359285,9390.467503,115682.621603,4412.535144
...,...,...,...,...,...
395,395,48785.709070,7090.431786,36000.287009,1580.499148
396,396,48758.430910,7060.511503,35970.746797,1569.863829
397,397,48750.243579,7075.218381,35935.586557,1557.634552
398,398,48753.799587,7087.687182,35906.368362,1588.026249


In [52]:
model = CatBoostRegressor(cat_features=['city', 'quality'], depth=6)
model.fit(X_train, y_train)

Learning rate set to 0.042022
0:	learn: 77380.7322751	total: 37.1ms	remaining: 37s
1:	learn: 75883.9086058	total: 79.2ms	remaining: 39.5s
2:	learn: 74611.2933604	total: 120ms	remaining: 40s
3:	learn: 73328.3468570	total: 162ms	remaining: 40.3s
4:	learn: 72309.8841751	total: 230ms	remaining: 45.8s
5:	learn: 71206.8252505	total: 294ms	remaining: 48.6s
6:	learn: 70164.6703119	total: 387ms	remaining: 55s
7:	learn: 69192.0138263	total: 434ms	remaining: 53.8s
8:	learn: 68368.6313032	total: 477ms	remaining: 52.5s
9:	learn: 67499.2760514	total: 519ms	remaining: 51.4s
10:	learn: 66576.5670695	total: 563ms	remaining: 50.6s
11:	learn: 65766.1131472	total: 610ms	remaining: 50.2s
12:	learn: 65055.1951657	total: 656ms	remaining: 49.8s
13:	learn: 64279.2670776	total: 703ms	remaining: 49.5s
14:	learn: 63538.4669478	total: 749ms	remaining: 49.2s
15:	learn: 62823.1284768	total: 792ms	remaining: 48.7s
16:	learn: 62326.3958011	total: 821ms	remaining: 47.5s
17:	learn: 61689.9807038	total: 864ms	remaining: 

CatBoostRegressor(cat_features=['city', 'quality'], depth=6, loss_function='RMSE')

In [53]:
ypred = model.predict(X_test)

print(f'MAE: {mean_absolute_error(y_test, ypred)}')
print(f'MAPE: {mean_absolute_percentage_error(y_test, ypred)}')

MAE: 27737.325347694034
MAPE: 0.30138301950363244


In [54]:
model_final = CatBoostRegressor(cat_features=['city', 'quality'], depth=6)
model_final.fit(X, y)

Learning rate set to 0.043531
0:	learn: 76402.7165393	total: 42.7ms	remaining: 42.7s
1:	learn: 75202.4156409	total: 84.7ms	remaining: 42.3s
2:	learn: 73812.2154323	total: 125ms	remaining: 41.5s
3:	learn: 72575.2081877	total: 193ms	remaining: 48.1s
4:	learn: 71466.7426619	total: 245ms	remaining: 48.7s
5:	learn: 70396.2144188	total: 333ms	remaining: 55.2s
6:	learn: 69405.5180603	total: 375ms	remaining: 53.2s
7:	learn: 68400.4390205	total: 411ms	remaining: 50.9s
8:	learn: 67392.4748290	total: 452ms	remaining: 49.7s
9:	learn: 66461.1522928	total: 493ms	remaining: 48.8s
10:	learn: 65498.3468566	total: 533ms	remaining: 47.9s
11:	learn: 64565.1335791	total: 575ms	remaining: 47.3s
12:	learn: 63776.6971888	total: 616ms	remaining: 46.8s
13:	learn: 63081.6683753	total: 659ms	remaining: 46.4s
14:	learn: 62382.7074382	total: 699ms	remaining: 45.9s
15:	learn: 61595.0136463	total: 737ms	remaining: 45.4s
16:	learn: 60941.7847988	total: 779ms	remaining: 45s
17:	learn: 60395.4138322	total: 815ms	remaini

CatBoostRegressor(cat_features=['city', 'quality'], depth=6, loss_function='RMSE')

<h1>Optimistic Evaluation</h1>

In [55]:
ypred = model_final.predict(X)

print(f'MAE: {mean_absolute_error(y, ypred)}')
print(f'MAPE: {mean_absolute_percentage_error(y, ypred)}')

MAE: 19355.708541373882
MAPE: 0.299645680922298


In [56]:
model_final.save_model('./GUI/airbnb_catboost.cmb')

In [57]:
X_test.to_csv('test.csv', index=False)